In [2]:
!pip install datasets sentence-transformers chromadb pandas

In [ ]:
import pandas as pd
import json

def load_jsonl_subset(file_path, max_rows):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= max_rows:
                break
            data.append(json.loads(line))
    return pd.DataFrame(data)

In [ ]:
from pathlib import Path
import urllib.request

META_URL = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/meta_categories/meta_Pet_Supplies.jsonl"
REVIEWS_URL = "https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/Pet_Supplies.jsonl"

META_FILE = Path("meta_Pet_Supplies.jsonl")
REVIEWS_FILE = Path("Pet_Supplies.jsonl")

def download_if_missing(url: str, target_path: Path):
    if target_path.exists() and target_path.stat().st_size > 0:
        print(f"Already exists: {target_path}")
        return
    print(f"Downloading {target_path.name} ...")
    urllib.request.urlretrieve(url, target_path)
    print(f"Saved: {target_path}")

download_if_missing(META_URL, META_FILE)
download_if_missing(REVIEWS_URL, REVIEWS_FILE)

df_meta = load_jsonl_subset(META_FILE, 5000)
df_reviews = load_jsonl_subset(REVIEWS_FILE, 20000)

print(f"df_meta shape: {df_meta.shape}")
print(f"df_reviews shape: {df_reviews.shape}")
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,4.0,Sticky stair riser tread thingies are utterly ...,"Tried to load photos, but none of my photos or...",[],B084SXF9Y8,B0BHTBS5RM,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1675529329744,0,True
1,1.0,Dangerous bc metal not properly coated! Rough ...,Where to begin? I’ve been trying to get the 2...,[{'small_image_url': 'https://m.media-amazon.c...,B000QFWCJ6,B0BJ16KKML,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1665513673369,4,True
2,3.0,Arrived damaged/dented/rusted,Unfortunately mine arrived damaged/dented whic...,[{'small_image_url': 'https://m.media-amazon.c...,B08B875X4H,B0BX76YVP9,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1656118516820,4,True
3,5.0,My pups love these!,My pups love these! It’s one of their favorit...,[],B01MFG9AG7,B0BM6V2SH8,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1589933838188,0,True
4,3.0,My pups refuse to eat them.,"Idk why, but my pups will not eat either flavo...",[],B00KRMMJV4,B0986BSRB1,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1587853303681,0,True


In [5]:
import pandas as pd
import numpy as np

# Clean description (handling lists)
df_meta['description'] = df_meta['description'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

# Clean categories (they are usually lists in this dataset, we want them as comma-separated strings)
if 'categories' in df_meta.columns:
    df_meta['categories'] = df_meta['categories'].apply(lambda x: ', '.join(x) if isinstance(x, list) else str(x))
else:
    df_meta['categories'] = "Pet Supplies" # Fallback if column is missing

# Clean missing brands
if 'store' in df_meta.columns:
    df_meta['brand'] = df_meta['store'].fillna('Unknown Brand')
else:
    df_meta['brand'] = "Unknown Brand"

# We strictly filter the reviews dataframe to ONLY include 4.0 and 5.0 ratings
good_reviews = df_reviews[df_reviews['rating'] >= 4.0].copy()

# Group only these positive reviews by product
positive_vibes = good_reviews.groupby('parent_asin')['text'].apply(
    lambda x: ' '.join(x.dropna().astype(str).head(5))
).reset_index()
positive_vibes.rename(columns={'text': 'positive_reviews'}, inplace=True)

# We do a LEFT JOIN. This means we keep all 5,000 products from our meta_data.
# If a product has good reviews, they get attached. If it doesn't, it gets a 'NaN' (null).
df_merged = pd.merge(df_meta, positive_vibes, on='parent_asin', how='left')

# THE FALLBACK: We replace those 'NaN' values with an empty string. 
# The product survives, but it just relies on its manufacturer description!
df_merged['positive_reviews'] = df_merged['positive_reviews'].fillna("")

# We inject structured data (Brand, Category) alongside the unstructured text!
df_merged['vibe_text'] = (
    "Title: " + df_merged['title'].astype(str) + ". " +
    "User Experience: " + df_merged['positive_reviews'] + " " +
    "Category: " + df_merged['categories'].astype(str) + ". " +
    "Brand: " + df_merged['brand'].astype(str) + ". " +
    "Description: " + df_merged['description'] 
)

# Let's peek at the first enriched text
print("\n--- Enriched 'Vibe Text' Example ---")
print(df_merged['vibe_text'].iloc[0][:600] + "...")


--- Enriched 'Vibe Text' Example ---
Title: Hurtta Pet Collection 14-Inch Padded Y-Harness, Pink. User Experience:  Category: Pet Supplies, Dogs, Collars, Harnesses & Leashes, Harnesses, Vest Harnesses. Brand: Hurtta. Description: Hurtta harnesses are suitable for active walks for all dogs, but they are especially recommended for dogs with back and neck problems. When the dog pulls on the leash, the close-fitting harness distributes pressure evenly across the chest, preventing damages to the dog’s vertebrae. Thanks to the wide padding and ergonomic design, the harness is comfortable and does not cause wear on the dog’s fur or ski...


In [6]:
import chromadb
from chromadb.utils import embedding_functions

# This creates a physical folder named 'pet_supplies_db' on your computer.
chroma_client = chromadb.PersistentClient(path="./pet_supplies_db")

# This downloads a lightweight, highly accurate model built specifically for semantic search
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# A collection is like a table in a standard SQL database
collection = chroma_client.get_or_create_collection(
    name="pet_vibe_search", 
    embedding_function=sentence_transformer_ef
)

# Safety check: drop any rows where vibe_text might accidentally be blank
df_clean = df_merged.dropna(subset=['vibe_text']).copy()

# Convert our Pandas columns into standard Python lists for ChromaDB
documents_list = df_clean['vibe_text'].tolist()
ids_list = df_clean['parent_asin'].astype(str).tolist()

# We save the Title, Brand, and Category as 'metadata'. 
# This way, when the AI finds a math match, we know what product to show the user!
metadata_list = [
    {
        "title": str(row['title']), 
        "brand": str(row.get('brand', 'Unknown')), 
        "category": str(row.get('categories', 'Pet Supplies'))
    } 
    for index, row in df_clean.iterrows()
]

# THE HEAVY LIFTING: Add everything to the database.
# ChromaDB automatically passes your text through the embedding model right here.
collection.add(
    documents=documents_list,
    metadatas=metadata_list,
    ids=ids_list
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
print("--- Pet Supplies Vibe Search ---")

user_query = "an indestructible chew toy for a large aggressive dog"

print(f"Searching for: '{user_query}'...\n")

# 2. The AI converts your sentence into math and finds the closest matches
results = collection.query(
    query_texts=[user_query],
    n_results=5 # We only want the top 5 best matches
)

# 3. Print the results out beautifully so it looks like a real search engine
for i in range(len(results['ids'][0])):
    title = results['metadatas'][0][i]['title']
    brand = results['metadatas'][0][i]['brand']
    category = results['metadatas'][0][i]['category']
    
    # Distance measures how far apart the math vectors are. 
    # Lower distance = closer match!
    distance = results['distances'][0][i] 
    
    print(f"Result #{i+1}")
    print(f"Title: {title}")
    print(f"Brand: {brand} | Category: {category}")
    print(f"Vector Distance: {distance:.4f}")
    print("-" * 50)

--- Pet Supplies Vibe Search ---
Searching for: 'an indestructible chew toy for a large aggressive dog'...

Result #1
Title: 3 Pack Dog Toys for Strong Aggressive chewers and Perfect for Small, Medium and Large Dogs.
Brand: Generic | Category: Pet Supplies, Dogs, Toys
Vector Distance: 0.2773
--------------------------------------------------
Result #2
Title: Dog Chew Toys for Aggressive Chewers, Rope Suction Cup Tug of War Toy, Toothbrush Molar Bite, Squeaky Ball, Dog Puzzle Teeth Cleaning and Food Dispensing for Small Medium and Large Dogs（America Blue
Brand: Kaleidoscope' Big World | Category: Pet Supplies, Dogs, Toys, Ropes
Vector Distance: 0.2774
--------------------------------------------------
Result #3
Title: Red Bone Toy with Rubber - Toy Chew Toy Ultra Durable Non-Toxic Pet Tooth Cleaning Interactive
Brand: Generic | Category: Pet Supplies, Dogs, Toys, Chew Toys
Vector Distance: 0.2899
--------------------------------------------------
Result #4
Title: MIGHTY- Massive-Nature-

In [8]:
# 1. Install the keyword search library directly in Jupyter
!pip install rank_bm25

import numpy as np
from rank_bm25 import BM25Okapi

# BM25 needs the text broken down into individual words (tokenized)
tokenized_corpus = [str(doc).lower().split(" ") for doc in documents_list]
bm25_engine = BM25Okapi(tokenized_corpus)

def hybrid_search(user_query, top_n=5):
    """Runs Vector Search and Keyword Search, then merges them with RRF."""
    
    # --- STEP 1: KEYWORD SEARCH (The Exact Words) ---
    tokenized_query = user_query.lower().split(" ")
    bm25_scores = bm25_engine.get_scores(tokenized_query)
    
    # Get the indices of the Top 20 highest-scoring exact matches
    top_bm25_indices = np.argsort(bm25_scores)[::-1][:20]
    keyword_ranked_ids = [ids_list[i] for i in top_bm25_indices]
    
    # --- STEP 2: VECTOR SEARCH (The Vibes) ---
    vector_results = collection.query(
        query_texts=[user_query],
        n_results=20
    )
    vector_ranked_ids = vector_results['ids'][0]
    
    # --- STEP 3: RECIPROCAL RANK FUSION (The Merger) ---
    rrf_scores = {}
    k_constant = 60 
    
    # Add RRF points from the Vector list
    for rank, doc_id in enumerate(vector_ranked_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_constant + rank + 1))
        
    # Add RRF points from the Keyword list
    for rank, doc_id in enumerate(keyword_ranked_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_constant + rank + 1))
        
    # --- STEP 4: SORT AND RETURN FINAL RESULTS ---
    # Sort the dictionary by highest RRF score
    final_sorted_results = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    
    # Grab the top N IDs to show the user
    top_final_ids = [doc_id for doc_id, score in final_sorted_results[:top_n]]
    
    return top_final_ids

print("\n--- 2. Testing the Hybrid Engine ---")
# Let's test it with a query that needs BOTH an exact brand match and a vibe match
test_query = "KONG durable chew toy for aggressive dog"
print(f"Searching for: '{test_query}'")

final_ids = hybrid_search(test_query, top_n=5)

# Print the readable metadata for our winning IDs
print("\n🏆 Top Hybrid Results:")
for i, target_id in enumerate(final_ids):
    # Find the row in our dataframe to get the title and brand
    row = df_clean[df_clean['parent_asin'] == target_id].iloc[0]
    print(f"Result #{i+1} | ID: {target_id}")
    print(f"Brand: {row.get('brand', 'Unknown')} | Title: {row['title']}")
    print("-" * 50)


--- 2. Testing the Hybrid Engine ---
Searching for: 'KONG durable chew toy for aggressive dog'

🏆 Top Hybrid Results:
Result #1 | ID: B0BZ18XZS2
Brand: KUMOUARTS | Title: Durble Dog Chew Toy Dog Bite Toy Doggy Toy Puppy Toy,Indestructible Dog Super Chewer Dog Toys Aggressive Chewers,Treat Toys,with Teeth Cleaning and Food Leakage Function for Medium/Small Dogs(Blue)
--------------------------------------------------
Result #2 | ID: B09JZR654G
Brand: KONG | Title: KONG ChewStix Tough Femur Dog Toy, Medium, Off-White
--------------------------------------------------
Result #3 | ID: B018R58M7W
Brand: Mavel | Title: Mavel Bond Toy for Dogs Puppies, Top Cool Rubber Dog Chew Toy, Best for Aggressive Chewers, Small Medium Large Dogs Breeds, Bacon Flavor
--------------------------------------------------
Result #4 | ID: B0B73XYMVK
Brand: Generic | Title: Red Bone Toy with Rubber - Toy Chew Toy Ultra Durable Non-Toxic Pet Tooth Cleaning Interactive
--------------------------------------------

In [9]:
#we can also perform a weighted hybrid search by giving more importance to either the vector or the keyword component.
tokenized_corpus = [str(doc).lower().split(" ") for doc in documents_list]
bm25_engine = BM25Okapi(tokenized_corpus)
def weighted_hybrid_search(user_query, top_n=5, alpha=0.5):
    """
    Runs Vector and Keyword Search, merging them with a weighted RRF.
    alpha = 1.0 (100% Vector/Vibe Search)
    alpha = 0.0 (100% Keyword/BM25 Search)
    alpha = 0.5 (Perfect 50/50 balance)
    """
    
    # --- STEP 1: KEYWORD SEARCH (BM25) ---
    tokenized_query = user_query.lower().split(" ")
    bm25_scores = bm25_engine.get_scores(tokenized_query)
    top_bm25_indices = np.argsort(bm25_scores)[::-1][:20]
    keyword_ranked_ids = [ids_list[i] for i in top_bm25_indices]
    
    # --- STEP 2: VECTOR SEARCH (ChromaDB) ---
    vector_results = collection.query(
        query_texts=[user_query],
        n_results=20
    )
    vector_ranked_ids = vector_results['ids'][0]
    
    # --- STEP 3: WEIGHTED RECIPROCAL RANK FUSION ---
    rrf_scores = {}
    k_constant = 60 
    
    # Calculate weights based on the alpha dial
    vector_weight = alpha
    keyword_weight = 1.0 - alpha
    
    # Add weighted RRF points from the Vector list
    for rank, doc_id in enumerate(vector_ranked_ids):
        base_score = 1.0 / (k_constant + rank + 1)
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (base_score * vector_weight)
        
    # Add weighted RRF points from the Keyword list
    for rank, doc_id in enumerate(keyword_ranked_ids):
        base_score = 1.0 / (k_constant + rank + 1)
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (base_score * keyword_weight)
        
    # --- STEP 4: SORT AND RETURN ---
    final_sorted_results = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    top_final_ids = [doc_id for doc_id, score in final_sorted_results[:top_n]]
    
    return top_final_ids

print("\n--- 2. Testing the Alpha Dial ---")
test_query = "KONG durable chew toy for aggressive dog"

# Test 1: Vibe Heavy (Alpha = 0.9)
print(f"\nSearch 1: '{test_query}' (Vibe Heavy / Alpha = 0.9)")
vibe_ids = weighted_hybrid_search(test_query, top_n=3, alpha=0.9)
for i, target_id in enumerate(vibe_ids):
    row = df_clean[df_clean['parent_asin'] == target_id].iloc[0]
    print(f"#{i+1} | Brand: {row.get('brand', 'Unknown')} | Title: {row['title']}")

# Test 2: Keyword Heavy (Alpha = 0.1)
print(f"\nSearch 2: '{test_query}' (Keyword Heavy / Alpha = 0.1)")
keyword_ids = weighted_hybrid_search(test_query, top_n=3, alpha=0.1)
for i, target_id in enumerate(keyword_ids):
    row = df_clean[df_clean['parent_asin'] == target_id].iloc[0]
    print(f"#{i+1} | Brand: {row.get('brand', 'Unknown')} | Title: {row['title']}")


--- 2. Testing the Alpha Dial ---

Search 1: 'KONG durable chew toy for aggressive dog' (Vibe Heavy / Alpha = 0.9)
#1 | Brand: KONG | Title: KONG ChewStix Tough Femur Dog Toy, Medium, Off-White
#2 | Brand: Mavel | Title: Mavel Bond Toy for Dogs Puppies, Top Cool Rubber Dog Chew Toy, Best for Aggressive Chewers, Small Medium Large Dogs Breeds, Bacon Flavor
#3 | Brand: Generic | Title: Red Bone Toy with Rubber - Toy Chew Toy Ultra Durable Non-Toxic Pet Tooth Cleaning Interactive

Search 2: 'KONG durable chew toy for aggressive dog' (Keyword Heavy / Alpha = 0.1)
#1 | Brand: KUMOUARTS | Title: Durble Dog Chew Toy Dog Bite Toy Doggy Toy Puppy Toy,Indestructible Dog Super Chewer Dog Toys Aggressive Chewers,Treat Toys,with Teeth Cleaning and Food Leakage Function for Medium/Small Dogs(Blue)
#2 | Brand: Generic | Title: Red Bone Toy with Rubber - Toy Chew Toy Ultra Durable Non-Toxic Pet Tooth Cleaning Interactive
#3 | Brand: SHINOE | Title: 2-in-1 Plush Dog Squeaky Toys, Cute Caterpillar Snuf

In [10]:
%%writefile app.py
import streamlit as st

# Set up the page
st.set_page_config(page_title="Pet Vibe Search", page_icon="🐾")

st.title("🐾 Pet Supplies 'Vibe' Search")
st.write("Welcome to the semantic search engine. Type what you are looking for below!")

# The Search Bar
user_query = st.text_input("What kind of vibe does your pet need?", placeholder="e.g., an indestructible toy for a pitbull")

# The Search Button
if st.button("Search"):
    if user_query:
        st.success(f"You searched for: {user_query}")
        st.write("*(In the next step, we will connect your Hybrid Search engine here!)*")
    else:
        st.warning("Please enter a search query.")

Overwriting app.py


In [11]:
# Save the enriched data so the web app can read it instantly
df_clean.to_csv("cleaned_pet_data.csv", index=False)
print("✅ Data saved for the web app!")

✅ Data saved for the web app!


In [12]:
%%writefile app.py
import streamlit as st
import pandas as pd
import chromadb
import numpy as np
from rank_bm25 import BM25Okapi
from chromadb.utils import embedding_functions
import google.generativeai as genai


@st.cache_data(ttl=3600, show_spinner=False)
def pick_gemini_model(api_key):
    """Pick an available Gemini model that supports generateContent."""
    genai.configure(api_key=api_key)
    preferred_models = [
        "gemini-2.0-flash",
        "gemini-1.5-flash",
        "gemini-1.5-pro",
    ]
    fallback = "gemini-1.5-flash"

    try:
        available = []
        for model in genai.list_models():
            methods = getattr(model, "supported_generation_methods", [])
            if "generateContent" in methods:
                available.append(model.name.replace("models/", ""))

        for name in preferred_models:
            if name in available:
                return name

        if available:
            return sorted(available)[0]
    except Exception:
        pass

    return fallback


@st.cache_resource
def load_search_system():
    df = pd.read_csv("cleaned_pet_data.csv")
    df["parent_asin"] = df["parent_asin"].astype(str)
    df_by_asin = df.set_index("parent_asin", drop=False)
    sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

    chroma_client = chromadb.PersistentClient(path="./pet_supplies_db")
    collection = chroma_client.get_collection(name="pet_vibe_search", embedding_function=sentence_transformer_ef)

    documents_list = df["vibe_text"].tolist()
    ids_list = df["parent_asin"].tolist()
    tokenized_corpus = [str(doc).lower().split(" ") for doc in documents_list]
    bm25 = BM25Okapi(tokenized_corpus)

    return df_by_asin, collection, bm25, ids_list


df_by_asin, collection, bm25_engine, ids_list = load_search_system()


def weighted_hybrid_search(user_query, top_n=5, alpha=0.5):
    tokenized_query = user_query.lower().split(" ")
    bm25_scores = bm25_engine.get_scores(tokenized_query)
    top_bm25_indices = np.argsort(bm25_scores)[::-1][:20]
    keyword_ranked_ids = [ids_list[i] for i in top_bm25_indices]

    vector_results = collection.query(query_texts=[user_query], n_results=20)
    vector_ranked_ids = vector_results["ids"][0]

    rrf_scores = {}
    k_constant = 60

    for rank, doc_id in enumerate(vector_ranked_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + ((1.0 / (k_constant + rank + 1)) * alpha)
    for rank, doc_id in enumerate(keyword_ranked_ids):
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + ((1.0 / (k_constant + rank + 1)) * (1.0 - alpha))

    final_sorted_results = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    return [doc_id for doc_id, score in final_sorted_results[:top_n]]


st.set_page_config(page_title="AI Pet Assistant", page_icon="🐾", layout="wide")

with st.sidebar:
    st.header("⚙️ App Settings")
    api_key = st.text_input("Enter Gemini API Key", type="password")
    st.markdown("Get a free key at [Google AI Studio](https://aistudio.google.com/app/apikey)")
    alpha_dial = st.slider("Search Tuning (Alpha)", 0.0, 1.0, 0.5, 0.1, help="0.0 = BM25, 1.0 = Vector")

st.title("🐾 AI Pet Shopping Assistant")
st.write("Ask me anything! I will search the database and give you a custom recommendation.")

user_query = st.text_input("What does your pet need today?")

if st.button("Search & Ask AI", type="primary"):
    if not user_query:
        st.warning("Please enter a search query.")
    else:
        with st.spinner("Searching database and thinking..."):
            top_ids = weighted_hybrid_search(user_query, top_n=5, alpha=alpha_dial)

            context_data = ""
            for target_id in top_ids:
                row = df_by_asin.loc[target_id]
                context_data += f"\n- Product: {row['title']} (Brand: {row.get('brand', 'Unknown')})\n  Details: {row['vibe_text'][:400]}...\n"

            ai_text = None
            ai_error = None

            if api_key:
                system_prompt = f"""
                You are a helpful and expert pet store assistant.
                A customer asked: \"{user_query}\"

                Based ONLY on the following products from our database, write a friendly,
                short recommendation explaining which product is best for them and why based on the vibes/reviews.

                Database Products:
                {context_data}
                """

                try:
                    model_name = pick_gemini_model(api_key)
                    model = genai.GenerativeModel(model_name)
                    response = model.generate_content(system_prompt)
                    ai_text = getattr(response, "text", "No text response received.")
                except Exception as e:
                    ai_error = str(e)

            if ai_text:
                st.success("✨ AI Recommendation")
                st.write(ai_text)
            elif api_key:
                st.warning("AI recommendation is temporarily unavailable.")
                if ai_error and "429" in ai_error:
                    st.info("Gemini API quota exceeded. Wait for reset or enable billing.")
                elif ai_error and "404" in ai_error:
                    st.info("Selected Gemini model is unavailable for your API version/project.")
                elif ai_error:
                    st.info(f"Gemini request failed: {ai_error}")
                st.markdown("Showing best matching products from hybrid search only.")
            else:
                st.info("No API key provided. Showing search results without AI summary.")

            st.divider()
            st.subheader("📚 Recommended Products")
            for i, target_id in enumerate(top_ids):
                row = df_by_asin.loc[target_id]
                st.markdown(f"**{i+1}. {row['title']}** ({row.get('brand', 'Unknown')})")

Overwriting app.py
